# 00. SurvFace 공식 프로토콜 및 run 동결

역할별 CSV와 설정을 immutable run에 고정합니다. gallery를 재표본추출하지 않고 공식 MAT/CSV 순서를 보존하며 `known_unknown=0`을 검증합니다.

| 모드 | 예상 시간 |
| --- | ---: |
| `EXECUTE_STAGE=False` | 1초 미만(preflight) |
| `EXECUTE_STAGE=True` | 약 10초~1분 |

> **진행/체크포인트/재시작**: 단계 시작·완료가 출력되고 완료 artifact가 체크포인트입니다. 이 노트북이 중단되면 부분 run을 이어 쓰지 말고 Kernel Restart 후 00 전체를 다시 실행하여 새 run을 만드십시오. 이후 노트북은 `runs/survface/active_run.json` 또는 `RONBUN_SURVFACE_RUN_DIR`만 사용합니다.


In [ ]:
# Step 1 실행 범위: 이 셀의 세 값만 바꾸고 Kernel Restart -> Run All
MODE = 'dev'             # 'dev' 또는 'real'
DATA_FRACTION = 0.10     # 0 < DATA_FRACTION <= 1
SEED = 42

import sys
from pathlib import Path

for _scope_root in (Path.cwd(), *Path.cwd().parents):
    if (_scope_root / 'research').is_dir():
        break
else:
    raise FileNotFoundError('D:/ronbun 내부에서 노트북을 실행하십시오.')
if str(_scope_root) not in sys.path:
    sys.path.insert(0, str(_scope_root))

from research.compression import PCA_SWEEP_DIMENSIONS
from research.experiments.scope import ExperimentScope

PCA_DIMENSIONS = (384, 256, 128, 64, 32)
PQ_SOURCE_DIMENSION = 512
if PCA_DIMENSIONS != tuple(PCA_SWEEP_DIMENSIONS):
    raise RuntimeError('노트북 PCA sweep과 공통 압축 정의가 다릅니다.')
EXPERIMENT_SCOPE = ExperimentScope(
    mode=MODE, data_fraction=DATA_FRACTION, seed=SEED
)
EXPERIMENT_SCOPE.as_dict()


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("D:/ronbun 내부에서 노트북을 실행하십시오.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import pandas as pd
import yaml
from IPython.display import display

from research.protocols import build_survface_official_protocol
from research.runtime import ProgressReporter, RunStore
from research.runtime.hashing import canonical_sha256

EXECUTE_STAGE = False
CONFIG_PATH = Path(os.environ.get(
    "RONBUN_SURVFACE_CONFIG",
    PROJECT_ROOT / "configs" / "experiments" / "survface_face_search.yaml",
))
DATA_DIR = PROJECT_ROOT / "data" / "interim" / "survface"
RUN_ROOT = PROJECT_ROOT / "runs" / "survface"
PROGRESS = ProgressReporter("SurvFace 00 protocol freeze", heartbeat_seconds=30)


## 1. 입력 preflight

`EXECUTE_STAGE=False`에서는 파일을 만들지 않습니다. config와 data preparation 출력 존재 여부만 확인합니다.


In [ ]:
input_paths = {
    "official_manifest": DATA_DIR / "official_manifest.csv",
    "gallery": DATA_DIR / "gallery.csv",
    "registered_probes": DATA_DIR / "registered_probes.csv",
    "unknown_unknown_probes": DATA_DIR / "unknown_unknown_probes.csv",
    "summary": DATA_DIR / "summary.json",
}
preflight = {
    "execute_stage": EXECUTE_STAGE,
    "config_path": str(CONFIG_PATH),
    "config_exists": CONFIG_PATH.is_file(),
    "inputs_exist": {role: path.is_file() for role, path in input_paths.items()},
    "run_root": str(RUN_ROOT),
}
display(pd.Series(preflight, name="value").to_frame())


## 2. 공식 역할 검증 및 run 생성

공식 gallery의 모든 행을 그대로 동결합니다. `enrollment_policy=official_all`, `enrollment_target=0`은 후속 template 단계에서 강제합니다.


In [ ]:
result = {"status": "not_executed", **preflight}
if EXECUTE_STAGE:
    PROGRESS.emit("입력 검증 시작", expected="약 10초~1분")
    missing = [name for name, exists in preflight["inputs_exist"].items() if not exists]
    if not CONFIG_PATH.is_file() or missing:
        raise FileNotFoundError({"config": str(CONFIG_PATH), "missing_inputs": missing})

    config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8")) or {}
    manifest = pd.read_csv(input_paths["official_manifest"])
    protocol = build_survface_official_protocol(manifest)
    if not protocol.known_unknown_probes.empty:
        raise ValueError("SurvFace 공식 프로토콜에는 known_unknown이 없어야 합니다.")

    frozen_frames = {
        "gallery": protocol.gallery,
        "registered_probes": protocol.registered_probes,
        "unknown_unknown_probes": protocol.unknown_unknown_probes,
    }
    for role, frame in frozen_frames.items():
        expected = list(range(len(frame)))
        if frame["protocol_index"].astype(int).tolist() != expected:
            raise ValueError(f"{role} protocol_index 순서가 변경되었습니다.")

    counts = {name: int(len(frame)) for name, frame in frozen_frames.items()}
    counts["known_unknown_probes"] = 0
    run = RunStore.create(
        experiment_name=str(config.get("run", {}).get("name", "survface_official")),
        config=config,
        root=RUN_ROOT,
        repo_root=PROJECT_ROOT,
    )
    run.record_input(CONFIG_PATH, role="experiment_config")
    for role, path in input_paths.items():
        run.record_input(path, role=role)

    with run.phase("00_official_protocol_and_run_freeze") as phase:
        suffix = f"A{phase.attempt:03d}"
        for role, frame in frozen_frames.items():
            source = phase.attempt_dir / f"{role}_{suffix}.csv"
            frame.to_csv(source, index=False, encoding="utf-8", lineterminator="\n")
            phase.publish_artifact(source)
        snapshot = {
            "protocol": "qmul-survface-v1-official-open-set-identification",
            "config_hash": canonical_sha256(config),
            "counts": counts,
            "known_unknown_count": 0,
            "gallery_template_policy": "official_all",
            "gallery_enrollment_target": 0,
            "order_column": "protocol_index",
        }
        source = phase.attempt_dir / f"protocol_snapshot_{suffix}.json"
        source.write_text(json.dumps(snapshot, ensure_ascii=False, indent=2), encoding="utf-8")
        phase.publish_artifact(source)
        phase.record_counts(**counts)
    PROGRESS.emit("00 완료", run_id=run.run_id, **counts)
    result = {
        "status": "completed",
        "run_id": run.run_id,
        "run_dir": str(run.run_dir),
        "active_run_pointer": str(RUN_ROOT / "active_run.json"),
        **counts,
    }
else:
    PROGRESS.emit("검토 모드 완료: run을 생성하지 않음", expected="1초 미만")
result


## 다음 단계

세 역할의 count와 `known_unknown_probes=0`을 확인한 뒤 01로 이동합니다. config나 역할별 CSV가 바뀌면 기존 run을 수정하지 말고 00부터 새 run을 만듭니다.
